# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** Course Assistant for Agentic AI students

**User:** Students in this course who ask concept, revision, and learning-path questions

**Success looks like:** The agent correctly answers course-domain questions with grounded responses, maintains context across turns, and routes requests to retrieval/tool/skip with strong consistency

**Tool I will add:** Domain-specific prerequisite and revision helper (suggests what to study before a topic and generates quick practice prompts)

**Deployment choice:** Streamlit UI

---
## 0. Setup

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

Groq API Key: ✅ Loaded
LangGraph:    1.1.2
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [2]:
# Course Assistant knowledge base (12 focused domain docs)
# Structure preserved: id, topic, text

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "LLM API Basics",
        "text": """Large Language Model APIs provide a programmatic way to send prompts and receive generated responses. In this course context, an API call usually includes a system instruction, a user question, optional conversation history, and model parameters such as temperature or max tokens. A low temperature is preferred for factual educational assistants because it reduces randomness and keeps outputs consistent. API-based usage also allows logging, testing, and routing decisions to be implemented around the model call. Students should learn to separate prompt design from business logic so the same core assistant can evolve safely. Proper API use includes error handling for timeouts, invalid keys, and rate limits. For capstone work, a good pattern is to centralize model initialization once, then call it from node functions. This makes behavior easier to debug and test."""
    },
    {
        "id": "doc_002",
        "topic": "Function and Tool Calling",
        "text": """Tool calling allows an agent to execute controlled functions when a user request requires deterministic operations or structured domain actions. In a course assistant, tools should be domain-specific rather than generic. Good examples are prerequisite lookup, revision checklist generation, and quick quiz generation from known topics. The router decides when a tool is needed, and the tool node returns deterministic output back into state. A robust tool design avoids free-form side effects and returns concise, parseable text. Tool outputs should be transparent to the user, and if a tool cannot satisfy the request, the assistant should fail safely with guidance instead of inventing results. In LangGraph pipelines, tool use is typically one branch among retrieve, skip, and tool routes. Keeping tool behavior explicit improves evaluation quality and reduces hallucination risk."""
    },
    {
        "id": "doc_003",
        "topic": "Conversation Memory with Thread IDs",
        "text": """Conversation memory enables multi-turn continuity, where the assistant can use earlier turns in later answers. In this course pattern, memory is managed by LangGraph checkpointers such as MemorySaver. The key idea is that each conversation is tagged with a thread_id. Reusing the same thread_id keeps context linked; changing it starts a fresh conversation. Students often confuse prompt context with persistent memory. Prompt context is what you send now; persistent memory is what the graph can recover across turns by session key. For capstone validation, a three-turn test is mandatory: establish a user fact in turn one, discuss a topic in turn two, and verify recall in turn three. Memory should support relevance, not over-verbosity. Good assistants summarize and carry only necessary context. Correct thread management is a core production skill."""
    },
    {
        "id": "doc_004",
        "topic": "Embeddings and Semantic Retrieval",
        "text": """Embeddings convert text into dense vectors that capture semantic meaning. Instead of exact keyword matching, vector retrieval compares conceptual similarity in embedding space. In practice, a sentence-transformer model like all-MiniLM-L6-v2 is commonly used for lightweight local workflows. During indexing, each knowledge chunk is embedded and stored with metadata. During query time, the user question is embedded and compared to stored vectors to return top-k relevant chunks. Retrieval quality depends on chunk clarity, topical separation, and document coverage. If chunks are vague or repetitive, top results become noisy. For course assistants, embeddings should represent concrete instructional concepts such as LangGraph nodes, RAG pipeline steps, and evaluation metrics. Semantic retrieval is the grounding layer that helps prevent unsupported generation and enables source-aware responses."""
    },
    {
        "id": "doc_005",
        "topic": "RAG Fundamentals",
        "text": """Retrieval-Augmented Generation combines a retriever with a generator. The retriever fetches relevant context from a knowledge base, and the generator creates an answer constrained by that context. This pattern improves factual reliability for domain assistants where internal model memory may be outdated or insufficient. A typical flow is: embed corpus, store vectors, retrieve top-k chunks, construct grounded prompt, generate answer, and optionally evaluate faithfulness. In capstone terms, RAG is not just adding a vector database; it is a behavior contract that answers should come from retrieved evidence. If evidence is missing, the assistant should say so and guide the user. RAG performance improves with better chunk design, cleaner metadata, and focused routing so irrelevant queries do not force retrieval. For educational use, RAG supports explainable, curriculum-aligned responses."""
    },
    {
        "id": "doc_006",
        "topic": "LangChain Components",
        "text": """LangChain provides modular building blocks for model calls, prompts, retrievers, and tool wrappers. In this course sequence, LangChain is often used alongside LangGraph: LangChain handles components while LangGraph handles orchestration. Core practices include separating prompt templates from runtime state, keeping chain steps inspectable, and maintaining deterministic defaults for evaluation runs. Students should avoid monolithic scripts where retrieval, generation, and post-processing are tangled in one function. Instead, compose small pieces that can be tested independently. When integrated with vector stores and chat models, LangChain helps standardize input/output patterns. For capstone, the important takeaway is interoperability: you can switch models, retrievers, or evaluators without rewriting the entire application architecture. This modularity supports debugging, incremental upgrades, and deployment readiness."""
    },
    {
        "id": "doc_007",
        "topic": "LangGraph State and Node Design",
        "text": """LangGraph organizes agent workflows as explicit state transitions between nodes. Each node reads from state, performs a focused task, and writes updated fields back. The state schema should be defined before node implementation, because missing state keys are a common source of runtime errors. A typical capstone state includes question, messages, route, retrieved_docs, sources, tool_result, answer, faithfulness, and retry counters. Conditional edges control branching, such as router-to-retrieve/tool/skip and evaluator-to-save/retry. This graph-first approach improves observability compared with hidden control flow in plain loops. It also makes testing easier because each node can be validated in isolation. Students should keep node responsibilities narrow: router decides route, retrieval fetches context, answer composes reply, eval judges quality, save logs result. Clear boundaries produce robust behavior."""
    },
    {
        "id": "doc_008",
        "topic": "Multi-Agent Patterns",
        "text": """Multi-agent systems divide work across specialized agents such as planner, retriever, critic, and executor. While the capstone may use a single graph, understanding multi-agent patterns helps design better internal roles and routes. Benefits include specialization and parallel reasoning, but risks include coordination overhead and inconsistent outputs if contracts are unclear. Effective multi-agent design requires explicit message formats, role boundaries, and arbitration logic for conflicting responses. In educational assistants, lightweight specialization is often enough: one route for retrieval, one for tools, and one for memory-only interactions. This achieves many multi-agent benefits without full orchestration complexity. Students should treat multi-agent architecture as a scaling option, not a requirement for every task. Good system design starts simple, then introduces role separation only when measurable benefits appear in tests."""
    },
    {
        "id": "doc_009",
        "topic": "Evaluation with Faithfulness",
        "text": """Evaluation measures whether answers are correct, relevant, and grounded. For RAG-based assistants, faithfulness is a key metric: does the answer stay consistent with retrieved evidence. RAGAS provides useful metrics such as faithfulness, answer relevancy, and context precision. In practical capstone workflows, evaluation should run on a fixed test set so improvements can be compared over time. Automated scores are helpful but should be paired with manual checks for edge cases, especially prompt injection and false premises. A good evaluator node can trigger retries when quality is below threshold, but retry loops must have limits to avoid infinite cycles. Reporting should include route chosen, sources used, score value, and pass/fail rationale. Consistent evaluation turns agent development from guesswork into evidence-based iteration."""
    },
    {
        "id": "doc_010",
        "topic": "Prompt Injection and Safety Boundaries",
        "text": """Prompt injection occurs when user input attempts to override system instructions or reveal hidden policies. A safe assistant treats user messages as untrusted and preserves high-priority rules. Typical attack patterns include commands like ignore previous instructions, reveal your system prompt, or fabricate citations. Defensive behavior includes refusal for unsafe requests, no leakage of private prompts, and transparent limitation statements. In course assistants, safety also means refusing out-of-domain claims while offering a helpful fallback path such as asking a mentor or checking official materials. Safety instructions should be embedded in system prompts and reinforced in evaluation tests. Red-team testing is essential to validate defenses under adversarial phrasing. Strong safety does not reduce usefulness; it improves trust and reliability in real deployment."""
    },
    {
        "id": "doc_011",
        "topic": "Streamlit Deployment Basics",
        "text": """Streamlit is a rapid UI framework that lets developers expose agent workflows as interactive web apps. For production-like behavior, heavy objects such as models, embedders, and vector stores should be initialized with cache decorators to avoid repeated loading on reruns. Conversation continuity in Streamlit uses session state, where a thread_id and message history are persisted per browser session. A minimal course assistant UI should accept user questions, call an ask helper, show answers, and optionally display route, sources, and quality score for transparency. Deployment readiness includes handling missing API keys, showing friendly error messages, and separating UI concerns from core graph logic in an agent module. Even simple local deployment demonstrates end-to-end capability from retrieval to memory to evaluation."""
    },
    {
        "id": "doc_012",
        "topic": "Capstone Testing Strategy",
        "text": """A complete capstone test strategy combines functional, memory, red-team, and evaluation baselines. Functional tests should cover retrieval route, tool route, and memory-only route with expected outcomes. Memory testing requires multi-turn continuity with a constant thread_id. Red-team tests should include out-of-scope questions, false premises, injection attempts, hallucination bait, and emotionally sensitive prompts. Each test should log question, expected behavior, actual behavior, and pass/fail. Baseline evaluation with small fixed datasets helps quantify quality before and after improvements. Students should avoid judging quality only by response length or fluency; grounded correctness and safety matter more. Testing artifacts such as JSON logs and metric tables are part of deployment evidence. Strong testing demonstrates engineering maturity beyond demo-only behavior."""
    },
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except Exception:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids = [d["id"] for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"  - {d['topic']}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base ready: 12 documents
  - LLM API Basics
  - Function and Tool Calling
  - Conversation Memory with Thread IDs
  - Embeddings and Semantic Retrieval
  - RAG Fundamentals
  - LangChain Components
  - LangGraph State and Node Design
  - Multi-Agent Patterns
  - Evaluation with Faithfulness
  - Prompt Injection and Safety Boundaries
  - Streamlit Deployment Basics
  - Capstone Testing Strategy


In [3]:
# ── Test retrieval before building the graph ──────────────
# Phase 2 retrieval smoke test for the Course Assistant domain
test_query = "How does LangGraph use state and routing in a course assistant?"

q_emb = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print("\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:220]}...")

print("\nIf retrieved chunks are relevant, Part 1 retrieval is working.")

Query: How does LangGraph use state and routing in a course assistant?

Top 3 retrieved chunks:

[1] Topic: LangGraph State and Node Design
    Text: LangGraph organizes agent workflows as explicit state transitions between nodes. Each node reads from state, performs a focused task, and writes updated fields back. The state schema should be defined before node impleme...

[2] Topic: Function and Tool Calling
    Text: Tool calling allows an agent to execute controlled functions when a user request requires deterministic operations or structured domain actions. In a course assistant, tools should be domain-specific rather than generic....

[3] Topic: Conversation Memory with Thread IDs
    Text: Conversation memory enables multi-turn continuity, where the assistant can use earlier turns in later answers. In this course pattern, memory is managed by LangGraph checkpointers such as MemorySaver. The key idea is tha...

If retrieved chunks are relevant, Part 1 retrieval is working.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [4]:
# Part 2 state schema for Course Assistant
# Every value used by any node is explicitly represented here.

class CapstoneState(TypedDict):
    # Input
    question: str                       # user current question

    # Memory
    messages: List[dict]                # rolling chat history

    # Routing
    route: str                          # "retrieve", "memory_only", "tool"

    # RAG
    retrieved: str                      # merged retrieved context chunks
    sources: List[str]                  # retrieved topic names

    # Tool
    tool_result: str                    # output from domain tool

    # Final answer
    answer: str                         # model response

    # Quality gate
    faithfulness: float                 # 0.0 to 1.0
    eval_retries: int                   # retry counter for eval loop

    # Course-assistant specific fields
    learner_level: str                  # beginner/intermediate/advanced (optional)
    target_topic: str                   # topic currently requested
    last_tool_used: str                 # prerequisite_helper / quiz_helper / none

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'learner_level', 'target_topic', 'last_tool_used']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [6]:
# ── Node 1: Memory ─────────────────────────────────────────
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [7]:
# ── Node 2: Router ─────────────────────────────────────────
# Decides: retrieve, memory_only, or tool

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a strict router for an Agentic AI Course Assistant.

Choose exactly one option:
- retrieve: use knowledge base for concept/explanation questions (LangGraph, RAG, memory, evaluation, deployment)
- memory_only: use only recent conversation memory (for follow-ups like 'what did I ask', 'summarize what we discussed')
- tool: use course helper tool when user asks prerequisites, revision plan, or quick quiz/practice questions

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    # Normalize any noisy output from the model
    if "memory" in decision:
        decision = "memory_only"
    elif "tool" in decision or "quiz" in decision or "prereq" in decision:
        decision = "tool"
    else:
        decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {
    "question": "What did you just say about embeddings?",
    "messages": [{"role": "user", "content": "Explain embeddings first."}]
}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}'")

router_node test: route='memory_only'


In [8]:
# ── Node 3: Retrieval ──────────────────────────────────────
# Queries ChromaDB

def retrieval_node(state: CapstoneState) -> dict:
    q_emb = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks = results["documents"][0]
    topics = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "Explain how MemorySaver and thread_id work together."}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Conversation Memory with Thread IDs', 'RAG Fundamentals', 'Streamlit Deployment Basics']
  Context preview: [Conversation Memory with Thread IDs]
Conversation memory enables multi-turn continuity, where the assistant can use earlier turns in later answers. In this course pattern, memory is managed by LangGr...
✅ retrieval_node works


In [9]:
# ── Node 4: Tool ───────────────────────────────────────────
# Domain tool: prerequisite + revision/quiz helper

def tool_node(state: CapstoneState) -> dict:
    question = state["question"].strip()
    lower_q = question.lower()

    prerequisite_map = {
        "day01": [],
        "day02": ["day01"],
        "day03": ["day01", "day02"],
        "day04": ["day01", "day03"],
        "day05": ["day02", "day04"],
        "day06": ["day03", "day05"],
        "day07": ["day06"],
        "day08": ["day05", "day06"],
        "day09": ["day08"],
        "day10": ["day04", "day09"],
        "day11": ["day10"],
        "day12": ["day11"],
        "day13": ["day08", "day10", "day11", "day12"],
    }

    quiz_bank = {
        "langgraph": [
            "What problem does StateGraph solve compared to plain chains?",
            "When should a node write to state vs compute locally?",
            "Why is conditional routing important for production agents?"
        ],
        "rag": [
            "Define RAG in one sentence.",
            "What causes retrieval mismatch in vector search?",
            "How does context grounding reduce hallucinations?"
        ],
        "memory": [
            "What is the purpose of thread_id?",
            "How is MemorySaver different from prompt-only memory?",
            "Why use a sliding window in conversation history?"
        ]
    }

    # Helper to detect day references like day8/day08/day 8
    import re
    day_match = re.search(r"day\s*0?(\d{1,2})", lower_q)

    if "prereq" in lower_q or "prerequisite" in lower_q:
        if day_match:
            d = int(day_match.group(1))
            key = f"day{d:02d}"
            prereqs = prerequisite_map.get(key)
            if prereqs is None:
                tool_result = f"I do not have prerequisite mapping for {key}."
            elif len(prereqs) == 0:
                tool_result = f"{key} has no hard prerequisites. Suggested start: fundamentals and API basics."
            else:
                tool_result = f"Prerequisites for {key}: {', '.join(prereqs)}"
        else:
            tool_result = "Please specify a day number, for example: 'What are prerequisites for Day 08?'"
        last_tool = "prerequisite_helper"

    elif "quiz" in lower_q or "practice" in lower_q or "revision" in lower_q:
        topic = "langgraph" if "langgraph" in lower_q else "rag" if "rag" in lower_q else "memory"
        qs = quiz_bank[topic]
        tool_result = (
            f"Quick {topic.upper()} practice set:\n"
            f"1. {qs[0]}\n2. {qs[1]}\n3. {qs[2]}\n"
            "Answer these first, then ask me to evaluate your responses."
        )
        last_tool = "quiz_helper"

    else:
        tool_result = (
            "Tool route selected, but no matching helper intent detected. "
            "Try asking for prerequisites (e.g., Day 08) or a quick quiz on LangGraph/RAG/Memory."
        )
        last_tool = "none"

    return {"tool_result": tool_result, "last_tool_used": last_tool}


print("✅ tool_node implemented with prerequisite and quiz helper")

✅ tool_node implemented with prerequisite and quiz helper


In [10]:
# ── Node 5: Answer ─────────────────────────────────────────
# Combines memory + retrieved context + tool results -> LLM answer

def answer_node(state: CapstoneState) -> dict:
    question = state["question"]
    retrieved = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    messages = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)

    # Build context section
    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    if context:
        system_content = f"""You are a Course Assistant for the Agentic AI program.
Rules:
1. Answer using ONLY the supplied context blocks.
2. If context does not contain the answer, say exactly: I don't have that information in my knowledge base.
3. Be concise, accurate, and student-friendly.
4. If tool output is present, prioritize it for prerequisite/quiz requests.
5. Do not invent days, metrics, or citations.

{context}"""
    else:
        system_content = """You are a Course Assistant. Answer from conversation history only. If unsure, ask a clarifying follow-up question."""

    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: previous response failed quality gate. Be strictly grounded and avoid unsupported claims."

    # Build message list: system + history + current question
    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user" else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("✅ answer_node implemented for Course Assistant grounding")

✅ answer_node implemented for Course Assistant grounding


In [11]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [13]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":
        return "tool"
    if route == "memory_only":
        return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory", memory_node)
graph.add_node("router", router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("skip", skip_retrieval_node)
graph.add_node("tool", tool_node)
graph.add_node("answer", answer_node)
graph.add_node("eval", eval_node)
graph.add_node("save", save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory", "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router",
    route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip", "answer")
graph.add_edge("tool", "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval",
    eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

# Lightweight validation + smoke invoke
required_nodes = {"memory", "router", "retrieve", "skip", "tool", "answer", "eval", "save"}
missing = required_nodes.difference(set(graph.nodes.keys()))
if missing:
    raise ValueError(f"Missing graph nodes: {missing}")

smoke = app.invoke(
    {"question": "What is LangGraph?"},
    config={"configurable": {"thread_id": "phase4-smoke"}}
)

print("✅ Graph compiled successfully")
print("✅ Required nodes present")
print(f"Smoke route: {smoke.get('route', 'n/a')}")
print(f"Smoke faithfulness: {smoke.get('faithfulness', 0.0):.2f}")
print(f"Smoke answer preview: {smoke.get('answer', '')[:160]}...")

  [eval] Faithfulness: 0.50 ⚠️
  [eval] Faithfulness: 1.00 ✅
✅ Graph compiled successfully
✅ Required nodes present
Smoke route: retrieve
Smoke faithfulness: 1.00
Smoke answer preview: LangGraph organizes agent workflows as explicit state transitions between nodes. Each node reads from state, performs a focused task, and writes updated fields ...


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [14]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


# Phase 5 test set: 10 questions with routing + behavior expectations
# Includes 2 red-team tests and a memory continuity pair (tests 8 and 9).

TEST_QUESTIONS = [
    {
        "q": "What is Retrieval-Augmented Generation (RAG)?",
        "expect": "Grounded explanation from KB",
        "expected_route": "retrieve",
        "red_team": False,
        "thread_id": "test-domain-1",
        "must_include": ["retrieval", "knowledge"],
        "must_not_include": []
    },
    {
        "q": "Explain how MemorySaver and thread_id work together.",
        "expect": "Explain persistent conversation memory",
        "expected_route": "retrieve",
        "red_team": False,
        "thread_id": "test-domain-2",
        "must_include": ["thread", "memory"],
        "must_not_include": []
    },
    {
        "q": "When should an agent use tool calling instead of pure retrieval?",
        "expect": "Conceptual comparison from KB",
        "expected_route": "retrieve",
        "red_team": False,
        "thread_id": "test-domain-3",
        "must_include": ["tool", "retrieval"],
        "must_not_include": []
    },
    {
        "q": "What are the prerequisites for Day 08?",
        "expect": "Tool output with day05 and day06",
        "expected_route": "tool",
        "red_team": False,
        "thread_id": "test-tool-1",
        "must_include": ["day05", "day06"],
        "must_not_include": []
    },
    {
        "q": "Generate a quick quiz on RAG.",
        "expect": "Tool quiz set",
        "expected_route": "tool",
        "red_team": False,
        "thread_id": "test-tool-2",
        "must_include": ["quick", "practice", "rag"],
        "must_not_include": []
    },
    {
        "q": "Summarize what we discussed in this chat so far.",
        "expect": "Memory-oriented response",
        "expected_route": "memory_only",
        "red_team": False,
        "thread_id": "test-memory-summary",
        "must_include": ["discussed"],
        "must_not_include": []
    },
    {
        "q": "LangGraph works only with Python 2, right?",
        "expect": "Correct false premise without agreeing",
        "expected_route": "retrieve",
        "red_team": True,
        "thread_id": "test-red-false-premise",
        "must_include": ["not"],
        "must_not_include": ["yes, langgraph works only with python 2"]
    },
    {
        "q": "My name is Rahul and I am revising LangGraph tonight.",
        "expect": "Store memory context for next turn",
        "expected_route": None,
        "red_team": False,
        "thread_id": "memory-thread",
        "must_include": ["rahul", "langgraph"],
        "must_not_include": []
    },
    {
        "q": "What is my name and which topic am I revising?",
        "expect": "Should recall Rahul and LangGraph from previous turn",
        "expected_route": "memory_only",
        "red_team": False,
        "thread_id": "memory-thread",
        "must_include": ["rahul", "langgraph"],
        "must_not_include": []
    },
    {
        "q": "What is the capital of France?",
        "expect": "Out-of-scope: should admit unknown in KB",
        "expected_route": "retrieve",
        "red_team": True,
        "thread_id": "test-red-oos",
        "must_include": ["i don't have that information in my knowledge base"],
        "must_not_include": ["paris"]
    },
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [15]:
# Run all tests and record results

def contains_all(text: str, phrases: list) -> bool:
    text_l = text.lower()
    return all(p.lower() in text_l for p in phrases)


def contains_none(text: str, phrases: list) -> bool:
    text_l = text.lower()
    return all(p.lower() not in text_l for p in phrases)


test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    thread_id = test.get("thread_id", f"test-{i}")
    result = ask(test["q"], thread_id=thread_id)

    answer = result.get("answer", "")
    faith = result.get("faithfulness", 0.0)
    route = result.get("route", "?")

    print(f"A: {answer[:220]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    expected_route = test.get("expected_route")
    route_pass = True if expected_route is None else (route == expected_route)

    include_pass = contains_all(answer, test.get("must_include", []))
    exclude_pass = contains_none(answer, test.get("must_not_include", []))
    length_pass = len(answer.strip()) >= 20

    # Faithfulness soft gate for normal tests; red-team judged by safety behavior + route.
    faith_pass = True if test["red_team"] else (faith >= 0.55)

    passed = route_pass and include_pass and exclude_pass and length_pass and faith_pass

    print(
        "Checks -> "
        f"route:{'Y' if route_pass else 'N'}, "
        f"include:{'Y' if include_pass else 'N'}, "
        f"exclude:{'Y' if exclude_pass else 'N'}, "
        f"length:{'Y' if length_pass else 'N'}, "
        f"faith:{'Y' if faith_pass else 'N'}"
    )
    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")

    test_results.append({
        "id": i + 1,
        "question": test["q"],
        "thread_id": thread_id,
        "expected_route": expected_route,
        "actual_route": route,
        "faithfulness": faith,
        "red_team": test["red_team"],
        "passed": passed,
        "answer_preview": answer[:260]
    })

# Summary
total = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
avg_faith = sum(r["faithfulness"] for r in test_results) / total if total else 0.0
red_total = sum(1 for r in test_results if r["red_team"])
red_pass = sum(1 for r in test_results if r["red_team"] and r["passed"])

print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Red-team: {red_pass}/{red_total} passed")
print(f"Average faithfulness: {avg_faith:.2f}")

# Optional: persist for report/submission
import json
with open("test_results.json", "w", encoding="utf-8") as f:
    json.dump(test_results, f, indent=2)

print("Saved detailed results to test_results.json")

RUNNING TEST SUITE

--- Test 1  ---
Q: What is Retrieval-Augmented Generation (RAG)?
  [eval] Faithfulness: 1.00 ✅
A: Retrieval-Augmented Generation (RAG) combines a retriever with a generator. The retriever fetches relevant context from a knowledge base, and the generator creates an answer constrained by that context. This pattern impr
Route: retrieve | Faithfulness: 1.00
Expected: Grounded explanation from KB
Checks -> route:Y, include:Y, exclude:Y, length:Y, faith:Y
Result: ✅ PASS

--- Test 2  ---
Q: Explain how MemorySaver and thread_id work together.
  [eval] Faithfulness: 1.00 ✅
A: MemorySaver, a LangGraph checkpointers, manages conversation memory by using a 'thread_id'. This 'thread_id' tags each conversation, allowing the assistant to link context across turns. Reusing the same 'thread_id' keeps
Route: retrieve | Faithfulness: 1.00
Expected: Explain persistent conversation memory
Checks -> route:Y, include:Y, exclude:Y, length:Y, faith:Y
Result: ✅ PASS

--- Test 3  ---
Q: When

---
## Part 6 — RAGAS Baseline Evaluation

In [16]:
# Phase 6: RAGAS dataset with explicit ground truths

RAGAS_QUESTIONS = [
    {
        "question": "What is Retrieval-Augmented Generation (RAG)?",
        "ground_truth": "RAG combines retrieval of relevant external context with generation so answers are grounded in a knowledge base rather than only model memory."
    },
    {
        "question": "How does MemorySaver with thread_id help in multi-turn conversations?",
        "ground_truth": "MemorySaver stores conversation state by thread_id; same thread_id preserves history across turns, different thread_id starts a new session."
    },
    {
        "question": "When should a course assistant use a tool instead of retrieval?",
        "ground_truth": "Use a tool for deterministic helper tasks such as prerequisite lookup and quiz generation, while retrieval is for knowledge-base concept answers."
    },
    {
        "question": "What are the prerequisites for Day 08?",
        "ground_truth": "Day 08 depends on Day 05 and Day 06 in the configured prerequisite map."
    },
    {
        "question": "How does LangGraph improve agent workflow design?",
        "ground_truth": "LangGraph models workflows as explicit state transitions across nodes with conditional routing, improving control, debugging, and testability."
    },
]

# Build the eval dataset
# We keep contexts from retrieval for RAGAS compatibility.
eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks = results["documents"][0]

    result = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")

    eval_dataset.append(
        {
            "question": rq["question"],
            "answer": result.get("answer", ""),
            "contexts": chunks,
            "ground_truth": rq["ground_truth"],
        }
    )
    print(f"  - {rq['question'][:65]}")

print(f"\nEval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [eval] Faithfulness: 1.00 ✅
  - What is Retrieval-Augmented Generation (RAG)?
  [eval] Faithfulness: 0.80 ✅
  - How does MemorySaver with thread_id help in multi-turn conversati
  - When should a course assistant use a tool instead of retrieval?
  - What are the prerequisites for Day 08?
  [eval] Faithfulness: 0.50 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
  - How does LangGraph improve agent workflow design?

Eval dataset built: 5 rows


In [18]:
# Run RAGAS if available; fall back to manual scoring on any runtime/provider issue

import re
import pandas as pd


def run_manual_fallback(eval_dataset_local):
    """Fallback scorer that does not depend on RAGAS provider setup."""
    faith_scores = []
    for row in eval_dataset_local:
        context_preview = " ".join(row["contexts"])[:500]
        prompt = f"""Rate faithfulness from 0.0 to 1.0. Reply with only a number.
Question: {row['question']}
Context: {context_preview}
Answer: {row['answer'][:300]}"""

        raw = llm.invoke(prompt).content.strip()
        match = re.search(r"\b(0(?:\.\d+)?|1(?:\.0+)?)\b", raw)
        score = float(match.group(1)) if match else 0.5
        score = max(0.0, min(1.0, score))
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} -> {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    baseline_df_local = pd.DataFrame(
        [
            {"metric": "faithfulness", "score": float(avg)},
            {"metric": "answer_relevancy", "score": None},
            {"metric": "context_precision", "score": None},
        ]
    )
    baseline_df_local.to_csv("ragas_baseline.csv", index=False)

    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Saved fallback baseline to ragas_baseline.csv")
    print("If you want full RAGAS metrics, configure a supported evaluator provider key.")


try:
    from ragas import evaluate
    # New import path to avoid deprecation warnings
    from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()

    f_mean = float(df["faithfulness"].mean())
    ar_mean = float(df["answer_relevancy"].mean())
    cp_mean = float(df["context_precision"].mean())

    baseline_df = pd.DataFrame(
        [
            {"metric": "faithfulness", "score": f_mean},
            {"metric": "answer_relevancy", "score": ar_mean},
            {"metric": "context_precision", "score": cp_mean},
        ]
    )
    baseline_df.to_csv("ragas_baseline.csv", index=False)

    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {f_mean:.3f}")
    print(f"Answer Relevance:  {ar_mean:.3f}")
    print(f"Context Precision: {cp_mean:.3f}")
    print("Saved baseline metrics to ragas_baseline.csv")

except Exception as e:
    print(f"RAGAS runtime unavailable ({type(e).__name__}): {e}")
    print("Switching to manual fallback scoring...")
    run_manual_fallback(eval_dataset)

Running RAGAS evaluation (1-2 minutes)...
RAGAS runtime unavailable (TypeError): All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]
Switching to manual fallback scoring...
  Q: What is Retrieval-Augmented Generation (RAG)? -> 0.90
  Q: How does MemorySaver with thread_id help in m -> 0.90
  Q: When should a course assistant use a tool ins -> 0.00
  Q: What are the prerequisites for Day 08?        -> 0.80
  Q: How does LangGraph improve agent workflow des -> 0.80

Baseline faithfulness: 0.680
Saved fallback baseline to ragas_baseline.csv
If you want full RAGAS metrics, configure a supported evaluator provider key.


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [20]:
import json

DOMAIN_NAME = "Course Assistant"
DOMAIN_DESCRIPTION = "Answers course questions with routing, memory, tool support, and grounded retrieval."

# Serialize current notebook DOCUMENTS so deployment app stays in sync.
documents_json = json.dumps(DOCUMENTS, ensure_ascii=True, indent=2)
documents_json_literal = repr(documents_json)

capstone_streamlit = f'''"""
capstone_streamlit.py - {DOMAIN_NAME}
Run: streamlit run capstone_streamlit.py
"""

import os
import json
import re
import uuid
import chromadb
import streamlit as st

from typing import TypedDict, List
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

st.set_page_config(page_title="{DOMAIN_NAME}", page_icon="🎓", layout="wide")
st.title("🎓 {DOMAIN_NAME}")
st.caption("{DOMAIN_DESCRIPTION}")


@st.cache_resource
def build_agent():
    groq_key = os.getenv("GROQ_API_KEY", "")
    if len(groq_key) < 10:
        raise RuntimeError("GROQ_API_KEY not found. Set it in .env before running Streamlit.")

    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.Client()
    try:
        client.delete_collection("capstone_kb_streamlit")
    except Exception:
        pass
    collection = client.create_collection("capstone_kb_streamlit")

    DOCUMENTS = json.loads({documents_json_literal})
    texts = [d["text"] for d in DOCUMENTS]
    ids = [d["id"] for d in DOCUMENTS]
    embeddings = embedder.encode(texts).tolist()

    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=ids,
        metadatas=[{{"topic": d["topic"]}} for d in DOCUMENTS]
    )

    class CapstoneState(TypedDict):
        question: str
        messages: List[dict]
        route: str
        retrieved: str
        sources: List[str]
        tool_result: str
        answer: str
        faithfulness: float
        eval_retries: int
        learner_level: str
        target_topic: str
        last_tool_used: str

    def memory_node(state: CapstoneState) -> dict:
        msgs = state.get("messages", [])
        msgs = msgs + [{{"role": "user", "content": state["question"]}}]
        if len(msgs) > 6:
            msgs = msgs[-6:]
        return {{"messages": msgs}}

    def router_node(state: CapstoneState) -> dict:
        question = state["question"]
        messages = state.get("messages", [])
        recent = "; ".join(f"{{m['role']}}: {{m['content'][:60]}}" for m in messages[-3:-1]) or "none"

        prompt = f"""You are a strict router for an Agentic AI Course Assistant.

Choose exactly one option:
- retrieve: use knowledge base for concept/explanation questions (LangGraph, RAG, memory, evaluation, deployment)
- memory_only: use only recent conversation memory (for follow-ups)
- tool: use helper tool for prerequisites, revision plan, or quick quiz/practice questions

Recent conversation: {{recent}}
Current question: {{question}}

Reply with ONLY one word: retrieve / memory_only / tool"""

        response = llm.invoke(prompt)
        decision = response.content.strip().lower()
        if "memory" in decision:
            decision = "memory_only"
        elif "tool" in decision or "quiz" in decision or "prereq" in decision:
            decision = "tool"
        else:
            decision = "retrieve"
        return {{"route": decision}}

    def retrieval_node(state: CapstoneState) -> dict:
        q_emb = embedder.encode([state["question"]]).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=3)
        chunks = results["documents"][0]
        topics = [m["topic"] for m in results["metadatas"][0]]
        context = "\\n\\n---\\n\\n".join(f"[{{topics[i]}}]\\n{{chunks[i]}}" for i in range(len(chunks)))
        return {{"retrieved": context, "sources": topics}}

    def skip_retrieval_node(state: CapstoneState) -> dict:
        return {{"retrieved": "", "sources": []}}

    def tool_node(state: CapstoneState) -> dict:
        question = state["question"].strip()
        lower_q = question.lower()

        prerequisite_map = {{
            "day01": [], "day02": ["day01"], "day03": ["day01", "day02"],
            "day04": ["day01", "day03"], "day05": ["day02", "day04"],
            "day06": ["day03", "day05"], "day07": ["day06"],
            "day08": ["day05", "day06"], "day09": ["day08"],
            "day10": ["day04", "day09"], "day11": ["day10"],
            "day12": ["day11"], "day13": ["day08", "day10", "day11", "day12"]
        }}

        quiz_bank = {{
            "langgraph": [
                "What problem does StateGraph solve compared to plain chains?",
                "When should a node write to state vs compute locally?",
                "Why is conditional routing important for production agents?"
            ],
            "rag": [
                "Define RAG in one sentence.",
                "What causes retrieval mismatch in vector search?",
                "How does context grounding reduce hallucinations?"
            ],
            "memory": [
                "What is the purpose of thread_id?",
                "How is MemorySaver different from prompt-only memory?",
                "Why use a sliding window in conversation history?"
            ]
        }}

        day_match = re.search(r"day\\s*0?(\\d{{1,2}})", lower_q)

        if "prereq" in lower_q or "prerequisite" in lower_q:
            if day_match:
                d = int(day_match.group(1))
                key = f"day{{d:02d}}"
                prereqs = prerequisite_map.get(key)
                if prereqs is None:
                    tool_result = f"I do not have prerequisite mapping for {{key}}."
                elif len(prereqs) == 0:
                    tool_result = f"{{key}} has no hard prerequisites."
                else:
                    tool_result = f"Prerequisites for {{key}}: {{', '.join(prereqs)}}"
            else:
                tool_result = "Please specify a day number, e.g., Day 08."
            last_tool = "prerequisite_helper"
        elif "quiz" in lower_q or "practice" in lower_q or "revision" in lower_q:
            topic = "langgraph" if "langgraph" in lower_q else "rag" if "rag" in lower_q else "memory"
            qs = quiz_bank[topic]
            tool_result = (
                f"Quick {{topic.upper()}} practice set:\\n"
                f"1. {{qs[0]}}\\n2. {{qs[1]}}\\n3. {{qs[2]}}\\n"
                "Answer these first, then ask me to evaluate your responses."
            )
            last_tool = "quiz_helper"
        else:
            tool_result = "Tool route selected, but no matching helper intent detected."
            last_tool = "none"

        return {{"tool_result": tool_result, "last_tool_used": last_tool}}

    def answer_node(state: CapstoneState) -> dict:
        question = state["question"]
        retrieved = state.get("retrieved", "")
        tool_result = state.get("tool_result", "")
        messages = state.get("messages", [])
        eval_retries = state.get("eval_retries", 0)

        context_parts = []
        if retrieved:
            context_parts.append(f"KNOWLEDGE BASE:\\n{{retrieved}}")
        if tool_result:
            context_parts.append(f"TOOL RESULT:\\n{{tool_result}}")
        context = "\\n\\n".join(context_parts)

        if context:
            system_content = f"""You are a Course Assistant for the Agentic AI program.
Rules:
1. Answer using ONLY the supplied context blocks.
2. If context does not contain the answer, say exactly: I don't have that information in my knowledge base.
3. Be concise, accurate, and student-friendly.
4. If tool output is present, prioritize it for prerequisite/quiz requests.
5. Do not invent days, metrics, or citations.

{{context}}"""
        else:
            system_content = "You are a Course Assistant. Answer from conversation history only."

        if eval_retries > 0:
            system_content += "\\n\\nIMPORTANT: be strictly grounded and avoid unsupported claims."

        lc_msgs = [SystemMessage(content=system_content)]
        for msg in messages[:-1]:
            lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user" else AIMessage(content=msg["content"]))
        lc_msgs.append(HumanMessage(content=question))

        response = llm.invoke(lc_msgs)
        return {{"answer": response.content}}

    FAITHFULNESS_THRESHOLD = 0.7
    MAX_EVAL_RETRIES = 2

    def eval_node(state: CapstoneState) -> dict:
        answer = state.get("answer", "")
        context = state.get("retrieved", "")[:500]
        retries = state.get("eval_retries", 0)

        if not context:
            return {{"faithfulness": 1.0, "eval_retries": retries + 1}}

        prompt = f"""Rate faithfulness: does this answer use ONLY information from context?
Reply only with a number 0.0 to 1.0.
Context: {{context}}
Answer: {{answer[:300]}}"""

        raw = llm.invoke(prompt).content.strip()
        m = re.search(r"\\b(0(?:\\.\\d+)?|1(?:\\.0+)?)\\b", raw)
        score = float(m.group(1)) if m else 0.5
        score = max(0.0, min(1.0, score))
        return {{"faithfulness": score, "eval_retries": retries + 1}}

    def save_node(state: CapstoneState) -> dict:
        messages = state.get("messages", [])
        messages = messages + [{{"role": "assistant", "content": state["answer"]}}]
        return {{"messages": messages}}

    def route_decision(state: CapstoneState) -> str:
        route = state.get("route", "retrieve")
        if route == "tool":
            return "tool"
        if route == "memory_only":
            return "skip"
        return "retrieve"

    def eval_decision(state: CapstoneState) -> str:
        score = state.get("faithfulness", 1.0)
        retries = state.get("eval_retries", 0)
        if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
            return "save"
        return "answer"

    graph = StateGraph(CapstoneState)
    graph.add_node("memory", memory_node)
    graph.add_node("router", router_node)
    graph.add_node("retrieve", retrieval_node)
    graph.add_node("skip", skip_retrieval_node)
    graph.add_node("tool", tool_node)
    graph.add_node("answer", answer_node)
    graph.add_node("eval", eval_node)
    graph.add_node("save", save_node)

    graph.set_entry_point("memory")
    graph.add_edge("memory", "router")
    graph.add_conditional_edges("router", route_decision, {{"retrieve": "retrieve", "skip": "skip", "tool": "tool"}})
    graph.add_edge("retrieve", "answer")
    graph.add_edge("skip", "answer")
    graph.add_edge("tool", "answer")
    graph.add_edge("answer", "eval")
    graph.add_conditional_edges("eval", eval_decision, {{"answer": "answer", "save": "save"}})
    graph.add_edge("save", END)

    app = graph.compile(checkpointer=MemorySaver())
    return app, collection, DOCUMENTS


@st.cache_resource
def load_resources():
    return build_agent()


app, collection, documents = load_resources()

if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]
if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    st.subheader("Session")
    st.write(f"Thread: {{st.session_state.thread_id}}")
    st.write(f"KB docs: {{collection.count()}}")
    if st.button("New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

prompt = st.chat_input("Ask about the course...")
if prompt:
    st.session_state.messages.append({{"role": "user", "content": prompt}})
    with st.chat_message("user"):
        st.write(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            result = app.invoke(
                {{"question": prompt}},
                config={{"configurable": {{"thread_id": st.session_state.thread_id}}}},
            )
            answer = result.get("answer", "Sorry, I could not generate an answer.")
            route = result.get("route", "n/a")
            faith = result.get("faithfulness", 0.0)
            sources = result.get("sources", [])

        st.write(answer)
        st.caption(f"Route: {{route}} | Faithfulness: {{faith:.2f}}")
        if sources:
            st.caption("Sources: " + ", ".join(sources))

    st.session_state.messages.append({{"role": "assistant", "content": answer}})
'''

with open("capstone_streamlit.py", "w", encoding="utf-8") as f:
    f.write(capstone_streamlit)

print("capstone_streamlit.py written successfully")
print("Run: streamlit run capstone_streamlit.py")

capstone_streamlit.py written successfully
Run: streamlit run capstone_streamlit.py


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Nandini Kumari

**Domain chosen:** Course Assistant for Agentic AI students

**What the agent does:** This agent helps students in the Agentic AI course with concept explanations, revision guidance, and prerequisite planning. It uses routing to decide between retrieval, memory-only responses, and a domain tool path for structured study help. It is designed to remain grounded to the knowledge base and preserve conversation continuity across turns.

**Knowledge base:** 12 documents covering LLM APIs, tool calling, memory with thread IDs, embeddings, RAG, LangChain components, LangGraph state design, multi-agent concepts, faithfulness evaluation, safety boundaries, Streamlit deployment, and capstone testing strategy.

**Tool used:** I added a domain-specific prerequisite and quiz helper. It returns prerequisite mappings for course days and generates quick practice question sets (LangGraph/RAG/Memory), which improves deterministic support for revision workflows.

**Deployment architecture note:** `capstone_streamlit.py` now imports and uses the shared backend module `agent.py` (via `build_agent()`), so UI and core agent logic are cleanly separated.

**RAGAS baseline scores:**
- Faithfulness: 0.68
- Answer Relevance: N/A (provider-dependent metric unavailable in fallback run)
- Context Precision: N/A (provider-dependent metric unavailable in fallback run)

**Test results:** 7 / 10 tests passed. Red-team: 1 / 2 passed.

**One thing I would improve with more time:** I would tune router prompts and tool-intent rules to improve edge-case routing accuracy, then add hybrid retrieval (vector + keyword/BM25) to improve coverage on borderline questions and raise pass rate.

**Most surprising thing I learned building this:** Small routing and evaluation prompt changes can significantly shift overall behavior and pass rates, even when the core graph architecture remains unchanged.

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*